<a href="https://colab.research.google.com/github/30Roll/escape/blob/main/%EB%86%80%EC%9D%B4%ED%84%B0_%EC%8B%9C%EC%84%A4%ED%98%84%ED%99%A9_%EB%B3%80%ED%99%98_%EA%B3%BC%EC%A0%95_%ED%9B%84_%EC%A0%84%EC%B2%98%EB%A6%AC_%EA%B3%BC%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
response = requests.get(
    "https://dapi.kakao.com/v2/local/search/address.json",
    headers={"Authorization": f"KakaoAK {KAKAO_KEY}"},
    params={"query": "서울특별시 동작구 상도로 369"},
    timeout=15
)

print("응답 코드:", response.status_code)
print("검색 결과:", len(response.json().get("documents", [])))

응답 코드: 200
검색 결과: 1


In [ ]:

# 원본 데이터 보존
df_original = df.copy()

print("전체 행 수:", len(df))

print("\n[결측값 개수]")
print(df.isna().sum())

print("\n[완전히 동일한 행의 중복 개수]")
print(df.duplicated().sum())

print("\n[시설번호 중복 개수]")
print(df["시설번호"].duplicated().sum())

print("\n[운영구분별 시설 수]")
print(df["운영구분"].value_counts(dropna=False))

print("\n[주소 결측 또는 공백 개수]")
print(
    df["주소"].isna().sum()
    + df["주소"].fillna("").astype(str).str.strip().eq("").sum()
    - df["주소"].isna().sum()
)

print("\n[숫자형 컬럼 음수 개수]")
for col in df.select_dtypes(include="number").columns:
    print(col, ":", (df[col] < 0).sum())


전체 행 수: 375

[결측값 개수]
연번       0
시설번호     0
놀이시설명    0
주소       0
설치일자     0
설치장소     0
운영구분     0
지역분류     0
민공구분     0
실내외구분    0
전화번호     0
dtype: int64

[완전히 동일한 행의 중복 개수]
0

[시설번호 중복 개수]
0

[운영구분별 시설 수]
운영구분
운영      374
이용금지      1
Name: count, dtype: int64

[주소 결측 또는 공백 개수]
0

[숫자형 컬럼 음수 개수]
연번 : 0
시설번호 : 0


In [ ]:

import requests
import pandas as pd
import time
import re
from tqdm.auto import tqdm
from google.colab import userdata

KAKAO_KEY = userdata.get("KAKAO_REST_API_KEY")

headers = {
    "Authorization": f"KakaoAK {KAKAO_KEY}"
}

valid_dongs = {
    "노량진1동", "노량진2동",
    "상도1동", "상도2동", "상도3동", "상도4동",
    "흑석동",
    "사당1동", "사당2동", "사당3동", "사당4동", "사당5동",
    "대방동", "신대방1동", "신대방2동"
}

def clean_address(address):
    if pd.isna(address) or not str(address).strip():
        return None

    # 괄호 안 아파트명 등 부가정보 제거
    return re.sub(r"\s*\([^)]*\)", "", str(address)).strip()

def convert_to_admin_dong(address):
    query = clean_address(address)

    if not query:
        return {"행정동": None, "변환상태": "주소없음"}

    try:
        # 1. 주소 -> 좌표
        r1 = requests.get(
            "https://dapi.kakao.com/v2/local/search/address.json",
            headers=headers,
            params={"query": query},
            timeout=15
        )
        r1.raise_for_status()

        docs = r1.json().get("documents", [])

        if not docs:
            return {"행정동": None, "변환상태": "주소검색실패"}

        # 여러 결과가 나오면 자동 확정하지 않음
        if len(docs) != 1:
            return {"행정동": None, "변환상태": "주소검색복수결과"}

        x, y = docs[0]["x"], docs[0]["y"]

        # 2. 좌표 -> 행정동
        r2 = requests.get(
            "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json",
            headers=headers,
            params={"x": x, "y": y},
            timeout=15
        )
        r2.raise_for_status()

        regions = r2.json().get("documents", [])

        matches = [
            region["region_3depth_name"]
            for region in regions
            if region.get("region_type") == "H"
            and region.get("region_2depth_name") == "동작구"
            and region.get("region_3depth_name") in valid_dongs
        ]

        if len(matches) == 1:
            return {
                "행정동": matches[0],
                "변환상태": "성공"
            }

        return {"행정동": None, "변환상태": "행정동확인실패"}

    except requests.RequestException:
        return {"행정동": None, "변환상태": "API오류"}

# 같은 주소는 한 번만 요청
unique_addresses = df["주소"].drop_duplicates()

address_map = {}

for address in tqdm(unique_addresses, desc="행정동 변환"):
    result = convert_to_admin_dong(address)
    address_map[address] = result
    time.sleep(0.15)

# 원본은 보존하고 새로운 데이터프레임 생성
df_mapped = df.copy()

df_mapped["행정동"] = df_mapped["주소"].map(
    lambda a: address_map[a]["행정동"]
)

df_mapped["변환상태"] = df_mapped["주소"].map(
    lambda a: address_map[a]["변환상태"]
)

print("\n[변환 상태별 시설 수]")
print(df_mapped["변환상태"].value_counts(dropna=False))

print("\n[행정동별 시설 수 - 변환 성공 건만]")
counts = (
    df_mapped[df_mapped["변환상태"] == "성공"]
    .groupby("행정동")
    .size()
    .reindex(sorted(valid_dongs), fill_value=0)
    .reset_index(name="확인된_놀이시설수")
)

display(counts)

print("\n원본 행 수:", len(df))
print("변환 후 행 수:", len(df_mapped))


행정동 변환:   0%|          | 0/276 [00:00<?, ?it/s]


[변환 상태별 시설 수]
변환상태
성공          366
주소검색실패        8
주소검색복수결과      1
Name: count, dtype: int64

[행정동별 시설 수 - 변환 성공 건만]


,행정동,확인된_놀이시설수
0,노량진1동,39
1,노량진2동,1
2,대방동,29
3,사당1동,7
4,사당2동,37
5,사당3동,31
6,사당4동,10
7,사당5동,12
8,상도1동,41
9,상도2동,30



원본 행 수: 375
변환 후 행 수: 375


In [ ]:

# 행정동 변환 실패 시설 확인
failed = df_mapped[
    df_mapped["변환상태"] != "성공"
].copy()

print("변환 실패 시설 수:", len(failed))

display(
    failed[
        ["시설번호", "놀이시설명", "주소",
         "지역분류", "운영구분", "변환상태"]
    ].reset_index(drop=True)
)

print("\n변환 실패 시설의 고유 주소 수:")
print(failed["주소"].nunique())


변환 실패 시설 수: 9


,시설번호,놀이시설명,주소,지역분류,운영구분,변환상태
0,522463,은하수어린이공원 놀이터,"서울특별시 동작구 여의대방로10길 38 (신대방동, 보라매롯데낙천대아파트) 신대방동...",서울 동작구 신대방동,운영,주소검색실패
1,522474,노량진공원(송학대-1) 놀이터,서울특별시 동작구 만양로13가길 28-3 (노량진동) 노량진동 221-97 노량진공...,서울 동작구 노량진동,운영,주소검색실패
2,543402,까치산마을공원 놀이터,서울특별시 동작구 사당로2가길 219 (사당동) 산32-56 까치산마을공원(사당5동...,서울 동작구 사당동,운영,주소검색실패
3,548627,사당자이아파트108동 놀이터,"서울특별시 동작구 사당로2가길 102 (사당동, 사당자이아파트) 105~8동놀이터",서울 동작구 사당동,운영,주소검색실패
4,552268,구립대방보듬이나눔이어린이집 실외놀이터,서울특별시 동작구 대방동15길 10-1 (대방동),서울 동작구 대방동,운영,주소검색실패
5,554249,효성 해링턴 플레이스 놀이터,서울특별시 동작구 상도로 209 (상도동) 효성 해링턴 플레이스 놀이터,서울 동작구 상도동,운영,주소검색실패
6,571653,구립한울어린이집 실외놀이터,서울특별시 동작구 보라매로9다길 17 (신대방동) 구립한울어린이집,서울 동작구 신대방동,운영,주소검색실패
7,1001653,생명숲교회 키즈놀이방,서울특별시 동작구 상도로 101-1 생명숲교회,서울 동작구 상도동,운영,주소검색복수결과
8,1006668,서울형키즈카페 시립 보라매공원점,서울특별시 동작구 보라매로5길 28 신대방동722 지상층,서울 동작구 신대방동,운영,주소검색실패



변환 실패 시설의 고유 주소 수:
9


In [ ]:

# 실패 시설의 주소 재검색 후보
retry_addresses = {
    522463: "서울특별시 동작구 여의대방로10길 38",
    522474: "서울특별시 동작구 만양로13가길 28-3",
    543402: "서울특별시 동작구 사당로2가길 219",
    548627: "서울특별시 동작구 사당로2가길 102",
    552268: "서울특별시 동작구 대방동15길 10-1",
    554249: "서울특별시 동작구 상도로 209",
    571653: "서울특별시 동작구 보라매로9다길 17",
    1001653: "서울특별시 동작구 상도로 101-1",
    1006668: "서울특별시 동작구 보라매로5길 28"
}

# 기존 데이터 보존
df_retry = df_mapped.copy()

# 재검색 결과 저장
retry_results = []

for facility_id, address in retry_addresses.items():
    result = convert_to_admin_dong(address)

    retry_results.append({
        "시설번호": facility_id,
        "재검색주소": address,
        "재검색행정동": result["행정동"],
        "재검색상태": result["변환상태"]
    })

retry_df = pd.DataFrame(retry_results)

display(retry_df)


,시설번호,재검색주소,재검색행정동,재검색상태
0,522463,서울특별시 동작구 여의대방로10길 38,신대방1동,성공
1,522474,서울특별시 동작구 만양로13가길 28-3,노량진1동,성공
2,543402,서울특별시 동작구 사당로2가길 219,사당5동,성공
3,548627,서울특별시 동작구 사당로2가길 102,사당5동,성공
4,552268,서울특별시 동작구 대방동15길 10-1,None,주소검색실패
5,554249,서울특별시 동작구 상도로 209,None,주소검색실패
6,571653,서울특별시 동작구 보라매로9다길 17,None,주소검색실패
7,1001653,서울특별시 동작구 상도로 101-1,상도3동,성공
8,1006668,서울특별시 동작구 보라매로5길 28,신대방2동,성공


In [ ]:

# 원본과 최초 변환 결과는 유지
df_review = df_mapped.copy()

# 재검색 성공 6건만 후보 행정동으로 기록
success_retry = retry_df[
    retry_df["재검색상태"] == "성공"
].copy()

mapping = dict(zip(
    success_retry["시설번호"],
    success_retry["재검색행정동"]
))

df_review["행정동_검토후보"] = df_review["행정동"]
df_review["검증상태"] = df_review["변환상태"].map(
    lambda x: "1차 API 변환 - 주소 검증 필요"
    if x == "성공" else "미확정"
)

mask = df_review["시설번호"].isin(mapping)

df_review.loc[mask, "행정동_검토후보"] = (
    df_review.loc[mask, "시설번호"].map(mapping)
)

df_review.loc[mask, "검증상태"] = (
    "재검색 API 변환 - 주소 검증 필요"
)

print("[검토 후보 행정동 확보]")
print(df_review["행정동_검토후보"].notna().sum())

print("\n[검증 상태]")
print(df_review["검증상태"].value_counts())

print("\n[남은 미확정 시설]")
display(
    df_review[
        df_review["행정동_검토후보"].isna()
    ][["시설번호", "놀이시설명", "주소"]]
)


[검토 후보 행정동 확보]
372

[검증 상태]
검증상태
1차 API 변환 - 주소 검증 필요     366
재검색 API 변환 - 주소 검증 필요      6
미확정                        3
Name: count, dtype: int64

[남은 미확정 시설]


,시설번호,놀이시설명,주소
254,552268,구립대방보듬이나눔이어린이집 실외놀이터,서울특별시 동작구 대방동15길 10-1 (대방동)
258,554249,효성 해링턴 플레이스 놀이터,서울특별시 동작구 상도로 209 (상도동) 효성 해링턴 플레이스 놀이터
300,571653,구립한울어린이집 실외놀이터,서울특별시 동작구 보라매로9다길 17 (신대방동) 구립한울어린이집


In [ ]:

import requests
import pandas as pd

targets = [
    (552268, "구립대방보듬이나눔이어린이집"),
    (554249, "효성 해링턴 플레이스 동작구"),
    (571653, "구립한울어린이집 동작구")
]

results = []

for facility_id, keyword in targets:
    response = requests.get(
        "https://dapi.kakao.com/v2/local/search/keyword.json",
        headers=headers,
        params={"query": keyword, "size": 10},
        timeout=15
    )

    response.raise_for_status()

    documents = response.json().get("documents", [])

    if not documents:
        results.append({
            "시설번호": facility_id,
            "검색어": keyword,
            "장소명": None,
            "도로명주소": None,
            "지번주소": None,
            "x": None,
            "y": None
        })

    for item in documents:
        results.append({
            "시설번호": facility_id,
            "검색어": keyword,
            "장소명": item.get("place_name"),
            "도로명주소": item.get("road_address_name"),
            "지번주소": item.get("address_name"),
            "x": item.get("x"),
            "y": item.get("y")
        })

keyword_results = pd.DataFrame(results)

display(keyword_results)


,시설번호,검색어,장소명,도로명주소,지번주소,x,y
0,552268,구립대방보듬이나눔이어린이집,None,None,None,None,None
1,554249,효성 해링턴 플레이스 동작구,상도효성해링턴플레이스아파트,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94200089814292,37.50579793751422
2,554249,효성 해링턴 플레이스 동작구,상도효성해링턴플레이스아파트 지하주차장,서울 동작구 상도로 207,서울 동작구 상도동 536,126.941644686863,37.5057355927269
3,554249,효성 해링턴 플레이스 동작구,해링턴플레이스문래아파트 (예정),,서울 영등포구 문래동3가 27-1,126.900122221406,37.518998490805
4,554249,효성 해링턴 플레이스 동작구,메가MGC커피 상도해링턴플레이스점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.9417546056311,37.50546084039922
5,554249,효성 해링턴 플레이스 동작구,올바른정육점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.941833786674,37.5054446613218
6,554249,효성 해링턴 플레이스 동작구,배스킨라빈스 장승배기역점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94191862047529,37.505431187993345
7,554249,효성 해링턴 플레이스 동작구,웰플란트치과의원,서울 동작구 상도로 207,서울 동작구 상도동 536,126.941790826308,37.505423016043
8,554249,효성 해링턴 플레이스 동작구,포이에마플라워,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94185630840063,37.50557081313221
9,554249,효성 해링턴 플레이스 동작구,폴리한의원,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94181221767624,37.505548266300046


In [ ]:

import requests
import pandas as pd

search_targets = {
    552268: [
        "동작구 대방동15길 10-1",
        "대방보듬이나눔이어린이집",
        "대방동 보듬이나눔이"
    ],
    554249: [
        "서울 동작구 상도로 207",
        "서울 동작구 상도동 536"
    ],
    571653: [
        "동작구 보라매로9다길 17",
        "한울어린이집 신대방동",
        "구립한울어린이집"
    ]
}

results = []

for facility_id, keywords in search_targets.items():
    for keyword in keywords:
        response = requests.get(
            "https://dapi.kakao.com/v2/local/search/keyword.json",
            headers=headers,
            params={"query": keyword, "size": 5},
            timeout=15
        )

        response.raise_for_status()

        for item in response.json().get("documents", []):
            if item.get("address_name", "").startswith("서울 동작구"):
                results.append({
                    "시설번호": facility_id,
                    "검색어": keyword,
                    "장소명": item.get("place_name"),
                    "도로명주소": item.get("road_address_name"),
                    "지번주소": item.get("address_name"),
                    "x": item.get("x"),
                    "y": item.get("y")
                })

additional_results = pd.DataFrame(results)

if additional_results.empty:
    print("추가 검색 결과가 없습니다.")
else:
    display(additional_results)


,시설번호,검색어,장소명,도로명주소,지번주소,x,y
0,554249,서울 동작구 상도로 207,레푸스 상도점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94185630770185,37.50557171413701
1,554249,서울 동작구 상도로 207,이두부야 상도점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94197521468016,37.50537265046976
2,554249,서울 동작구 상도로 207,상도효성해링턴플레이스아파트,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94200089814292,37.50579793751422
3,554249,서울 동작구 상도로 207,메가MGC커피 상도해링턴플레이스점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.9417546056311,37.50546084039922
4,554249,서울 동작구 상도로 207,올바른정육점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.941833786674,37.5054446613218
5,554249,서울 동작구 상도동 536,레푸스 상도점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94185630770185,37.50557171413701
6,554249,서울 동작구 상도동 536,이두부야 상도점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94197521468016,37.50537265046976
7,554249,서울 동작구 상도동 536,상도효성해링턴플레이스아파트,서울 동작구 상도로 207,서울 동작구 상도동 536,126.94200089814292,37.50579793751422
8,554249,서울 동작구 상도동 536,메가MGC커피 상도해링턴플레이스점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.9417546056311,37.50546084039922
9,554249,서울 동작구 상도동 536,올바른정육점,서울 동작구 상도로 207,서울 동작구 상도동 536,126.941833786674,37.5054446613218


In [ ]:

import requests

# 카카오 검색 결과의 아파트 좌표
x = 126.94200089814292
y = 37.50579793751422

response = requests.get(
    "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json",
    headers=headers,
    params={"x": x, "y": y},
    timeout=15
)

response.raise_for_status()

for region in response.json().get("documents", []):
    if region.get("region_type") == "H":
        print("행정구:", region["region_2depth_name"])
        print("행정동:", region["region_3depth_name"])
        print("행정동 코드:", region["code"])


행정구: 동작구
행정동: 상도2동
행정동 코드: 1159054000


In [ ]:

targets = [552268, 571653]

display(
    df_mapped.loc[
        df_mapped["시설번호"].isin(targets),
        ["시설번호", "놀이시설명", "주소",
         "설치장소", "지역분류", "운영구분"]
    ]
)


,시설번호,놀이시설명,주소,설치장소,지역분류,운영구분
254,552268,구립대방보듬이나눔이어린이집 실외놀이터,서울특별시 동작구 대방동15길 10-1 (대방동),어린이집,서울 동작구 대방동,운영
300,571653,구립한울어린이집 실외놀이터,서울특별시 동작구 보라매로9다길 17 (신대방동) 구립한울어린이집,어린이집,서울 동작구 신대방동,운영


In [ ]:

import requests
import pandas as pd

search_cases = {
    552268: [
        "서울 동작구 대방동15길 10-1",
        "동작구 대방동15길 10",
        "대방동15길 10-1"
    ],
    571653: [
        "서울 동작구 보라매로9다길 17",
        "동작구 보라매로9다길 17",
        "보라매로9다길 17"
    ]
}

results = []

for facility_id, addresses in search_cases.items():
    for address in addresses:
        response = requests.get(
            "https://dapi.kakao.com/v2/local/search/address.json",
            headers=headers,
            params={"query": address},
            timeout=15
        )

        response.raise_for_status()
        docs = response.json().get("documents", [])

        if not docs:
            results.append({
                "시설번호": facility_id,
                "검색어": address,
                "검색결과": "없음",
                "검색된주소": None,
                "x": None,
                "y": None
            })

        for item in docs:
            results.append({
                "시설번호": facility_id,
                "검색어": address,
                "검색결과": "있음",
                "검색된주소": item.get("address_name"),
                "x": item.get("x"),
                "y": item.get("y")
            })

address_retry_results = pd.DataFrame(results)

display(address_retry_results)


,시설번호,검색어,검색결과,검색된주소,x,y
0,552268,서울 동작구 대방동15길 10-1,없음,None,None,None
1,552268,동작구 대방동15길 10,있음,서울 동작구 대방동15길 10,126.924411452563,37.5029323456162
2,552268,대방동15길 10-1,없음,None,None,None
3,571653,서울 동작구 보라매로9다길 17,없음,None,None,None
4,571653,동작구 보라매로9다길 17,없음,None,None,None
5,571653,보라매로9다길 17,없음,None,None,None


In [ ]:

import requests

x = 126.924411452563
y = 37.5029323456162

response = requests.get(
    "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json",
    headers=headers,
    params={"x": x, "y": y},
    timeout=15
)

response.raise_for_status()

for region in response.json().get("documents", []):
    if region.get("region_type") == "H":
        print("행정구:", region["region_2depth_name"])
        print("행정동:", region["region_3depth_name"])
        print("행정동 코드:", region["code"])


행정구: 동작구
행정동: 대방동
행정동 코드: 1159066000


In [ ]:

# 공식 도로명주소로 재검색
new_address = "서울특별시 동작구 보라매로9다길 15"

result = convert_to_admin_dong(new_address)

print("시설번호: 571653")
print("시설명: 구립한울어린이집 실외놀이터")
print("재검색 주소:", new_address)
print("행정동:", result["행정동"])
print("변환 상태:", result["변환상태"])


시설번호: 571653
시설명: 구립한울어린이집 실외놀이터
재검색 주소: 서울특별시 동작구 보라매로9다길 15
행정동: 신대방2동
변환 상태: 성공


In [ ]:

import pandas as pd

# 기존 데이터 보존
df_final = df_review.copy()

# 1차 및 재검색으로 확보한 행정동 후보
df_final["분석용_행정동"] = df_final["행정동_검토후보"]

# 추가로 확보한 3건의 행정동 후보
extra_dongs = {
    554249: "상도2동",
    552268: "대방동",
    571653: "신대방2동"
}

# 원본 주소와 보정 주소를 구분
df_final["주소_보정후보"] = pd.NA

corrected_addresses = {
    554249: "서울특별시 동작구 상도로 207",
    552268: "서울특별시 동작구 대방동15길 10",
    571653: "서울특별시 동작구 보라매로9다길 15"
}

for facility_id, dong in extra_dongs.items():
    mask = df_final["시설번호"] == facility_id

    df_final.loc[mask, "분석용_행정동"] = dong
    df_final.loc[mask, "주소_보정후보"] = corrected_addresses[facility_id]
    df_final.loc[mask, "검증상태"] = "주소 보정 후보 - 검증 필요"

# 행 수 및 누락 여부 확인
print("전체 시설 수:", len(df_final))
print("행정동 후보 확보:", df_final["분석용_행정동"].notna().sum())
print("행정동 미확정:", df_final["분석용_행정동"].isna().sum())

assert len(df_final) == 375
assert df_final["분석용_행정동"].notna().all()

# 운영 중인 시설만 별도 집계
operating = df_final[
    df_final["운영구분"] == "운영"
].copy()

summary = (
    operating.groupby("분석용_행정동")
    .size()
    .reset_index(name="운영중_놀이시설수_잠정")
    .sort_values("운영중_놀이시설수_잠정", ascending=False)
)

print("\n[15개 행정동별 운영 중 놀이시설 수 - 잠정]")
display(summary)


전체 시설 수: 375
행정동 후보 확보: 375
행정동 미확정: 0

[15개 행정동별 운영 중 놀이시설 수 - 잠정]


,분석용_행정동,운영중_놀이시설수_잠정
14,흑석동,47
8,상도1동,41
0,노량진1동,40
4,사당2동,37
5,사당3동,31
2,대방동,30
9,상도2동,30
13,신대방2동,27
12,신대방1동,22
11,상도4동,21


In [ ]:

from google.colab import drive
import os

drive.mount("/content/drive")

folder = "/content/drive/MyDrive/동작구_어린이놀이시설_분석"
os.makedirs(folder, exist_ok=True)

# 시설별 전체 데이터
df_final.to_csv(
    f"{folder}/어린이놀이시설_행정동변환_검토본.csv",
    index=False,
    encoding="utf-8-sig"
)

# 행정동별 집계 결과
summary.to_csv(
    f"{folder}/행정동별_운영중_놀이시설수_잠정.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Google Drive 저장 완료!")


Mounted at /content/drive
Google Drive 저장 완료!


In [ ]:

from google.colab import files

# ① 시설 375개 전체 데이터
df_final.to_csv(
    "동작구_어린이놀이시설_행정동변환결과.csv",
    index=False,
    encoding="utf-8-sig"
)

# ② 행정동별 운영 중 놀이시설 수
summary.to_csv(
    "동작구_15개행정동별_놀이시설수.csv",
    index=False,
    encoding="utf-8-sig"
)

# 두 파일 다운로드
files.download("동작구_어린이놀이시설_행정동변환결과.csv")
files.download("동작구_15개행정동별_놀이시설수.csv")

print("CSV 파일 생성 완료!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSV 파일 생성 완료!


놀이터 시설 현황 행정동 기준으로 변환 과정 종료


------------------------------------------------
전처리 과정 시작

In [ ]:

import pandas as pd
from google.colab import files
from io import BytesIO

# 두 파일 업로드
uploaded = files.upload()

def read_csv_auto(content):
    for enc in ["utf-8-sig", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(BytesIO(content), encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError("파일 인코딩 확인 필요")

data = {}

for filename, content in uploaded.items():
    data[filename] = read_csv_auto(content)
    print(f"\n파일명: {filename}")
    print("크기:", data[filename].shape)
    print("컬럼:", data[filename].columns.tolist())
    display(data[filename].head(3))

# 파일명을 기준으로 구분
population_files = [
    name for name in data
    if "주민등록인구" in name
]

facility_files = [
    name for name in data
    if "행정동" in name and "시설" in name
]

if len(population_files) != 1 or len(facility_files) != 1:
    raise ValueError(
        "주민등록인구 CSV 1개와 "
        "시설별 행정동 변환 CSV 1개를 업로드해주세요."
    )

pop_raw = data[population_files[0]].copy()
facility_raw = data[facility_files[0]].copy()

print("\n[원본 품질 점검]")

for name, frame in [
    ("주민등록인구", pop_raw),
    ("어린이놀이시설", facility_raw)
]:
    print(f"\n{name}")
    print("전체 행 수:", len(frame))
    print("결측 셀 수:", int(frame.isna().sum().sum()))
    print("중복 행 수:", int(frame.duplicated().sum()))

    print("숫자형 컬럼별 음수:")
    for col in frame.select_dtypes(include="number"):
        print(col, int((frame[col] < 0).sum()))


Saving 동작구_어린이놀이시설_행정동변환결과.csv to 동작구_어린이놀이시설_행정동변환결과 (2).csv
Saving 서울시_주민등록인구_260204_dl1008.csv to 서울시_주민등록인구_260204_dl1008 (2).csv

파일명: 동작구_어린이놀이시설_행정동변환결과 (2).csv
크기: (375, 17)
컬럼: ['연번', '시설번호', '놀이시설명', '주소', '설치일자', '설치장소', '운영구분', '지역분류', '민공구분', '실내외구분', '전화번호', '행정동', '변환상태', '행정동_검토후보', '검증상태', '분석용_행정동', '주소_보정후보']


,연번,시설번호,놀이시설명,주소,설치일자,설치장소,운영구분,지역분류,민공구분,실내외구분,전화번호,행정동,변환상태,행정동_검토후보,검증상태,분석용_행정동,주소_보정후보
0,1,229,한강현대아파트 관리소앞,"서울특별시 동작구 현충로 151 (흑석동, 한강현대아파트)",2008-10-28,주택단지,운영,서울 동작구 흑석동,민간,실외,02-812-6812,흑석동,성공,흑석동,1차 API 변환 - 주소 검증 필요,흑석동,NaN
1,2,484,신한토탈아파트 어린이놀이터1,"서울특별시 동작구 등용로14길 34 (대방동, 신한토탈아파트)",2009-02-04,주택단지,운영,서울 동작구 대방동,민간,실외,02-824-8314,대방동,성공,대방동,1차 API 변환 - 주소 검증 필요,대방동,NaN
2,3,487,신한토탈아파트 어린이놀이터2,"서울특별시 동작구 등용로14길 34 (대방동, 신한토탈아파트)",2009-02-05,주택단지,운영,서울 동작구 대방동,민간,실외,02-824-8314,대방동,성공,대방동,1차 API 변환 - 주소 검증 필요,대방동,NaN



파일명: 서울시_주민등록인구_260204_dl1008 (2).csv
크기: (19, 105)
컬럼: ['동별(1)', '동별(2)', '동별(3)', '2026 2/4', '2026 2/4.1', '2026 2/4.2', '2026 2/4.3', '2026 2/4.4', '2026 2/4.5', '2026 2/4.6', '2026 2/4.7', '2026 2/4.8', '2026 2/4.9', '2026 2/4.10', '2026 2/4.11', '2026 2/4.12', '2026 2/4.13', '2026 2/4.14', '2026 2/4.15', '2026 2/4.16', '2026 2/4.17', '2026 2/4.18', '2026 2/4.19', '2026 2/4.20', '2026 2/4.21', '2026 2/4.22', '2026 2/4.23', '2026 2/4.24', '2026 2/4.25', '2026 2/4.26', '2026 2/4.27', '2026 2/4.28', '2026 2/4.29', '2026 2/4.30', '2026 2/4.31', '2026 2/4.32', '2026 2/4.33', '2026 2/4.34', '2026 2/4.35', '2026 2/4.36', '2026 2/4.37', '2026 2/4.38', '2026 2/4.39', '2026 2/4.40', '2026 2/4.41', '2026 2/4.42', '2026 2/4.43', '2026 2/4.44', '2026 2/4.45', '2026 2/4.46', '2026 2/4.47', '2026 2/4.48', '2026 2/4.49', '2026 2/4.50', '2026 2/4.51', '2026 2/4.52', '2026 2/4.53', '2026 2/4.54', '2026 2/4.55', '2026 2/4.56', '2026 2/4.57', '2026 2/4.58', '2026 2/4.59', '2026 2/4.60', '2026 2/4.61

,동별(1),동별(2),동별(3),2026 2/4,2026 2/4.1,2026 2/4.2,2026 2/4.3,2026 2/4.4,2026 2/4.5,2026 2/4.6,...,2026 2/4.92,2026 2/4.93,2026 2/4.94,2026 2/4.95,2026 2/4.96,2026 2/4.97,2026 2/4.98,2026 2/4.99,2026 2/4.100,2026 2/4.101
0,동별(1),동별(2),동별(3),합계,합계,합계,합계,합계,합계,합계,...,합계,합계,합계,합계,합계,합계,합계,합계,합계,합계
1,동별(1),동별(2),동별(3),소계,0세,1세,2세,3세,4세,5세,...,91세,92세,93세,94세,95세,96세,97세,98세,99세,100세 이상+
2,합계,소계,소계,9289813,47447,41798,37057,38917,40473,42841,...,11865,8890,6687,5064,3467,2772,2113,1446,933,1469



[원본 품질 점검]

주민등록인구
전체 행 수: 19
결측 셀 수: 0
중복 행 수: 0
숫자형 컬럼별 음수:

어린이놀이시설
전체 행 수: 375
결측 셀 수: 384
중복 행 수: 0
숫자형 컬럼별 음수:
연번 0
시설번호 0


In [ ]:

import pandas as pd

print("========== ① 주민등록인구 ==========")

# 실제 연령 컬럼이 표시된 두 번째 머리글 행
print("\n[연령 컬럼 구조]")
for i in range(3, 15):
    print(
        i,
        pop_raw.columns[i],
        "→",
        pop_raw.iloc[1, i]
    )

print("\n[동별 행 목록]")
display(
    pop_raw[
        ["동별(1)", "동별(2)", "동별(3)"]
    ]
)

print("\n========== ② 놀이시설 ==========")

print("\n[컬럼별 결측값]")
print(facility_raw.isna().sum())

print("\n[운영 상태]")
print(
    facility_raw["운영구분"].value_counts(
        dropna=False
    )
)

print("\n[행정동 후보 개수]")
print(
    facility_raw["분석용_행정동"].nunique(
        dropna=True
    )
)

print("\n[검증 상태별 시설 수]")
print(
    facility_raw["검증상태"].value_counts(
        dropna=False
    )
)

print("\n[시설번호 중복]")
print(
    facility_raw["시설번호"].duplicated().sum()
)


========== ① 주민등록인구 ==========

[연령 컬럼 구조]
3 2026 2/4 → 소계
4 2026 2/4.1 → 0세
5 2026 2/4.2 → 1세
6 2026 2/4.3 → 2세
7 2026 2/4.4 → 3세
8 2026 2/4.5 → 4세
9 2026 2/4.6 → 5세
10 2026 2/4.7 → 6세
11 2026 2/4.8 → 7세
12 2026 2/4.9 → 8세
13 2026 2/4.10 → 9세
14 2026 2/4.11 → 10세

[동별 행 목록]


,동별(1),동별(2),동별(3)
0,동별(1),동별(2),동별(3)
1,동별(1),동별(2),동별(3)
2,합계,소계,소계
3,합계,동작구,소계
4,합계,동작구,노량진1동
5,합계,동작구,노량진2동
6,합계,동작구,상도1동
7,합계,동작구,상도2동
8,합계,동작구,상도3동
9,합계,동작구,상도4동



========== ② 놀이시설 ==========

[컬럼별 결측값]
연번            0
시설번호          0
놀이시설명         0
주소            0
설치일자          0
설치장소          0
운영구분          0
지역분류          0
민공구분          0
실내외구분         0
전화번호          0
행정동           9
변환상태          0
행정동_검토후보      3
검증상태          0
분석용_행정동       0
주소_보정후보     372
dtype: int64

[운영 상태]
운영구분
운영      374
이용금지      1
Name: count, dtype: int64

[행정동 후보 개수]
15

[검증 상태별 시설 수]
검증상태
1차 API 변환 - 주소 검증 필요     366
재검색 API 변환 - 주소 검증 필요      6
주소 보정 후보 - 검증 필요           3
Name: count, dtype: int64

[시설번호 중복]
0


In [ ]:

import pandas as pd
import numpy as np

# ① 원본 복사
pop_clean = pop_raw.copy()

print("[1] 전처리 전 행 수:", len(pop_clean))

# ② 머리글 2행과 합계 2행 제외
pop_clean = pop_clean[
    (pop_clean["동별(2)"] == "동작구") &
    (pop_clean["동별(3)"] != "소계")
].copy()

print("[2] 동작구 행정동 추출 후:", len(pop_clean))

# ③ 필요한 컬럼만 선택
age_columns = pop_raw.columns[9:14].tolist()

pop_clean = pop_clean[
    ["동별(3)", pop_raw.columns[3]] + age_columns
].copy()

# ④ 컬럼 이름 변경
pop_clean.columns = [
    "행정동",
    "전체인구",
    "5세",
    "6세",
    "7세",
    "8세",
    "9세"
]

# ⑤ 숫자형 변환
numeric_cols = [
    "전체인구", "5세", "6세",
    "7세", "8세", "9세"
]

for col in numeric_cols:
    pop_clean[col] = pd.to_numeric(
        pop_clean[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# ⑥ 음수 오류값 점검
negative_counts = (pop_clean[numeric_cols] < 0).sum()

print("\n[3] 음수 오류값 개수")
print(negative_counts)

# 음수는 오류값으로 판단해 NaN으로 표시
pop_clean[numeric_cols] = pop_clean[numeric_cols].mask(
    pop_clean[numeric_cols] < 0
)

# ⑦ 결측값 및 중복 점검
print("\n[4] 컬럼별 결측값")
print(pop_clean.isna().sum())

print("\n[5] 행정동 중복 수")
print(pop_clean["행정동"].duplicated().sum())

# ⑧ 결측값이 없는 경우에만 인구 지표 계산
if pop_clean[numeric_cols].isna().any().any():
    print("\n결측값 발견: 보간·대체 방법 결정 후 계산 필요")
else:
    pop_clean["5~9세_인구"] = pop_clean[
        ["5세", "6세", "7세", "8세", "9세"]
    ].sum(axis=1)

    pop_clean["5~9세_인구비중(%)"] = (
        pop_clean["5~9세_인구"] /
        pop_clean["전체인구"] * 100
    ).round(2)

    print("\n[6] 전처리 결과")
    display(pop_clean)

print("\n최종 행 수:", len(pop_clean))


[1] 전처리 전 행 수: 19
[2] 동작구 행정동 추출 후: 15

[3] 음수 오류값 개수
전체인구    0
5세      0
6세      0
7세      0
8세      0
9세      0
dtype: int64

[4] 컬럼별 결측값
행정동     0
전체인구    0
5세      0
6세      0
7세      0
8세      0
9세      0
dtype: int64

[5] 행정동 중복 수
0

[6] 전처리 결과


,행정동,전체인구,5세,6세,7세,8세,9세,5~9세_인구,5~9세_인구비중(%)
4,노량진1동,33529,147,107,126,142,140,662,1.97
5,노량진2동,7615,10,8,12,12,14,56,0.74
6,상도1동,43968,150,161,164,216,207,898,2.04
7,상도2동,26716,150,179,197,183,211,920,3.44
8,상도3동,23331,65,69,65,84,81,364,1.56
9,상도4동,28647,110,122,107,135,136,610,2.13
10,흑석동,29844,192,219,230,249,276,1166,3.91
11,사당1동,22100,39,47,36,53,55,230,1.04
12,사당2동,27594,109,138,153,163,164,727,2.63
13,사당3동,23223,152,156,173,180,173,834,3.59



최종 행 수: 15


In [ ]:

import pandas as pd
import numpy as np

# ① 원본 복사
facility_clean = facility_raw.copy()

print("[1] 전처리 전 행 수:", len(facility_clean))

# ② 핵심 컬럼 점검
required_cols = [
    "시설번호", "놀이시설명", "주소",
    "운영구분", "분석용_행정동"
]

print("\n[2] 핵심 컬럼 결측값")
print(facility_clean[required_cols].isna().sum())

print("\n[3] 시설번호 중복 수")
print(facility_clean["시설번호"].duplicated().sum())

print("\n[4] 운영구분")
print(facility_clean["운영구분"].value_counts(dropna=False))

# ③ 분석용 행정동 공백 정리
facility_clean["분석용_행정동"] = (
    facility_clean["분석용_행정동"]
    .astype("string")
    .str.strip()
)

# ④ 운영 중인 시설만 추출
facility_operating = facility_clean[
    facility_clean["운영구분"] == "운영"
].copy()

print("\n[5] 운영 중 시설 추출 후 행 수:",
      len(facility_operating))

# ⑤ 행정동별 운영 중 시설 수 집계
facility_by_dong = (
    facility_operating
    .groupby("분석용_행정동")
    .agg(
        운영중_놀이시설수=("시설번호", "count")
    )
    .reset_index()
    .rename(columns={"분석용_행정동": "행정동"})
)

print("\n[6] 행정동별 시설 수")
display(facility_by_dong)

# ⑥ 집계 검증
print("\n[7] 집계 검증")
print("운영 중 시설 수:", len(facility_operating))
print("행정동별 시설 수 합계:",
      facility_by_dong["운영중_놀이시설수"].sum())
print("집계된 행정동 수:", len(facility_by_dong))

# 검증 실패 시 중단
assert len(facility_clean) == 375
assert len(facility_operating) == 374
assert facility_clean["시설번호"].duplicated().sum() == 0
assert facility_operating["분석용_행정동"].notna().all()
assert facility_by_dong["운영중_놀이시설수"].sum() == 374
assert len(facility_by_dong) == 15

print("\n놀이시설 전처리 및 집계 완료!")


[1] 전처리 전 행 수: 375

[2] 핵심 컬럼 결측값
시설번호       0
놀이시설명      0
주소         0
운영구분       0
분석용_행정동    0
dtype: int64

[3] 시설번호 중복 수
0

[4] 운영구분
운영구분
운영      374
이용금지      1
Name: count, dtype: int64

[5] 운영 중 시설 추출 후 행 수: 374

[6] 행정동별 시설 수


,행정동,운영중_놀이시설수
0,노량진1동,40
1,노량진2동,1
2,대방동,30
3,사당1동,7
4,사당2동,37
5,사당3동,31
6,사당4동,10
7,사당5동,14
8,상도1동,41
9,상도2동,30



[7] 집계 검증
운영 중 시설 수: 374
행정동별 시설 수 합계: 374
집계된 행정동 수: 15

놀이시설 전처리 및 집계 완료!


In [ ]:

import pandas as pd
import numpy as np

# ① 주민등록인구를 기준으로 놀이시설 데이터 결합
merged_df = pd.merge(
    pop_clean,
    facility_by_dong,
    on="행정동",
    how="left",
    validate="one_to_one",
    indicator=True
)

# ② 결합 결과 검증
print("[1] 결합 상태")
print(merged_df["_merge"].value_counts())

print("\n[2] 결합 전후 행 수")
print("인구 데이터:", len(pop_clean))
print("시설 집계 데이터:", len(facility_by_dong))
print("결합 데이터:", len(merged_df))

print("\n[3] 결합되지 않은 행정동")
display(
    merged_df.loc[
        merged_df["_merge"] != "both",
        ["행정동", "_merge"]
    ]
)

# ③ 결합 검증
assert len(merged_df) == 15
assert (merged_df["_merge"] == "both").all()
assert merged_df["운영중_놀이시설수"].notna().all()
assert merged_df["운영중_놀이시설수"].sum() == 374

# ④ 보조지표 계산
merged_df["놀이시설1곳당_어린이수"] = (
    merged_df["5~9세_인구"] /
    merged_df["운영중_놀이시설수"].replace(0, np.nan)
).round(2)

# ⑤ 어린이 인구 비중 기준 순위 계산
merged_df["어린이인구비중_순위"] = (
    merged_df["5~9세_인구비중(%)"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# ⑥ 결합 확인용 컬럼 제거 및 정렬
merged_df = (
    merged_df
    .drop(columns="_merge")
    .sort_values(
        ["5~9세_인구비중(%)", "행정동"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# ⑦ 최종 분석 테이블
result_cols = [
    "어린이인구비중_순위",
    "행정동",
    "전체인구",
    "5~9세_인구",
    "5~9세_인구비중(%)",
    "운영중_놀이시설수",
    "놀이시설1곳당_어린이수"
]

print("\n[4] 최종 결합 결과")
display(merged_df[result_cols])

print("\n결합 완료!")


[1] 결합 상태
_merge
both          15
left_only      0
right_only     0
Name: count, dtype: int64

[2] 결합 전후 행 수
인구 데이터: 15
시설 집계 데이터: 15
결합 데이터: 15

[3] 결합되지 않은 행정동


,행정동,_merge



[4] 최종 결합 결과


,어린이인구비중_순위,행정동,전체인구,5~9세_인구,5~9세_인구비중(%),운영중_놀이시설수,놀이시설1곳당_어린이수
0,1,흑석동,29844,1166,3.91,47,24.81
1,2,사당3동,23223,834,3.59,31,26.90
2,3,상도2동,26716,920,3.44,30,30.67
3,4,신대방2동,21991,581,2.64,27,21.52
4,5,사당2동,27594,727,2.63,37,19.65
5,6,신대방1동,22795,577,2.53,22,26.23
6,7,사당5동,13631,314,2.30,14,22.43
7,8,대방동,33466,752,2.25,30,25.07
8,9,상도4동,28647,610,2.13,21,29.05
9,10,상도1동,43968,898,2.04,41,21.90



결합 완료!


In [ ]:

from google.colab import files

# ① 주민등록인구 전처리 결과
pop_clean.to_csv(
    "01_주민등록인구_전처리결과.csv",
    index=False,
    encoding="utf-8-sig"
)

# ② 놀이시설 전처리 결과 (운영 중 374건)
facility_operating.to_csv(
    "02_놀이시설_전처리결과.csv",
    index=False,
    encoding="utf-8-sig"
)

# ③ 행정동별 시설 집계
facility_by_dong.to_csv(
    "03_행정동별_놀이시설집계.csv",
    index=False,
    encoding="utf-8-sig"
)

# ④ 최종 결합 데이터
merged_df.to_csv(
    "04_최종_결합분석데이터.csv",
    index=False,
    encoding="utf-8-sig"
)

# 파일 다운로드
files.download("01_주민등록인구_전처리결과.csv")
files.download("02_놀이시설_전처리결과.csv")
files.download("03_행정동별_놀이시설집계.csv")
files.download("04_최종_결합분석데이터.csv")

print("전처리 및 결합 CSV 4개 생성 완료!")


NameError: name 'pop_clean' is not defined

최대값,초최대값,초